# Week 6, Day 6: Forge: Week 6 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w006/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", np.linalg.eigvals(np.array([[3.0, 1.0], [0.0, 2.0]])))
print("2.", np.linalg.eigvals(np.array([[0.0, -2.0], [2.0, 0.0]])))
print("3.", 20 / (1 - 0.8), "ms")

### Exercise: Part A: where does an E-I circuit oscillate?

For the linearised E-I circuit $J = \begin{pmatrix} w_{EE} - 1 & -w_{EI} \\ w_{IE} & -1 - w_{II} \end{pmatrix}$ with $w_{IE} = 2$ and $w_{II} = 0.5$, fill `max_re[i, j]` with the largest real part of the eigenvalues at $w_{EI}$ = `wEI[i]` and $w_{EE}$ = `wEE[j]`, and `is_complex[i, j]` with whether they are complex. Then describe the three regions in the map.

In [ ]:
import matplotlib.pyplot as plt
def J_ei(wEE, wEI, wIE=2.0, wII=0.5):
    return np.array([[wEE - 1, -wEI], [wIE, -1 - wII]])

wEE = np.linspace(0, 4, 41)
wEI = np.linspace(0, 4, 41)
max_re = np.zeros((41, 41))
is_complex = np.zeros((41, 41), dtype=bool)
for i in range(41):
    for j in range(41):
        ...

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(np.where(max_re < 0, np.where(is_complex, 1, 0), 2), origin="lower", extent=[0, 4, 0, 4], cmap="viridis", aspect="auto")
ax.contour(wEE, wEI, max_re, levels=[0], colors="w")
ax.set_xlabel("recurrent excitation w_EE"); ax.set_ylabel("inhibition onto E, w_EI"); plt.show()
print("stable and oscillating:", np.mean((max_re < 0) & is_complex).round(2), " unstable:", np.mean(max_re >= 0).round(2))

In [ ]:
import numpy as np
def test_shapes():
    "max_re and is_complex are 41 by 41 (rows: w_EI, columns: w_EE)"
    assert max_re.shape == (41, 41) and is_complex.shape == (41, 41)

def test_values():
    "entries match the eigenvalues of J at sample points"
    for i, j in [(0, 0), (10, 30), (40, 5), (25, 25)]:
        lam = np.linalg.eigvals(J_ei(wEE[j], wEI[i]))
        assert np.isclose(max_re[i, j], lam.real.max())
        assert is_complex[i, j] == bool(np.abs(lam.imag).max() > 1e-12)

def test_regions():
    "the map contains stable oscillation, stable non-oscillation and instability"
    stable = max_re < 0
    assert np.any(stable & is_complex) and np.any(stable & ~is_complex) and np.any(~stable)

test_shapes()
test_values()
test_regions()
print("All checks passed")

In [ ]:
rng = np.random.default_rng(1996)
N = 40
m = rng.uniform(0.5, 1.5, N)
R = rng.normal(size=(N, N))
W0 = 0.9 * np.outer(m, m) / (m @ m) + 0.15 * (R + R.T) / np.sqrt(2 * N)
tau, dt = 0.1, 0.001

def run(W, x0, n_steps, readout):
    """Euler simulation; returns the readout (dot product with a vector) at each step."""
    x = x0.copy(); out = [readout @ x]
    for _ in range(n_steps):
        x = x + dt / tau * (-x + W @ x)
        out.append(readout @ x)
    return np.array(out)
print("symmetric:", np.allclose(W0, W0.T))

### Exercise: Part B: tune it into a line attractor

Find the top eigenvalue `lam1` and eigenvector `v1` of `W0` (flip `v1`'s sign if needed so it sums to a positive number). Scale the weights to make `W_tuned` with top eigenvalue exactly 1, and check that activity along `v1` is held: `held` is the readout over 2 s from a noisy start. Then mistune by 2% (`0.98 * W_tuned`), start exactly on `v1`, and estimate the time constant `tau_meas` from the slope of the log readout between 0.5 s and 3 s. Compare it with `tau_pred`.

In [ ]:
lams, Q = np.linalg.eigh(W0)
lam1, v1 = ..., ...
v1 = v1 * np.sign(v1.sum())
W_tuned = ...

held = run(W_tuned, v1 + 0.3 * rng.normal(size=N), 2000, v1)

decay = run(0.98 * W_tuned, v1, 3000, v1)
t = np.arange(3001) * dt
slope = ...                                 # fit log(decay) against t for t from 0.5 s on
tau_meas = ...
tau_pred = ...

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(lams, "o", ms=3); ax[0].axhline(1, ls="--", color="0.5"); ax[0].set_title("eigenvalues of W0"); ax[0].set_xlabel("index")
ax[1].plot(t[:2001], held, label="tuned"); ax[1].plot(t, decay, label="mistuned by 2%")
ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("activity along v1"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()
print(f"top eigenvalue {lam1:.3f}; predicted {tau_pred:.2f} s, measured {tau_meas:.2f} s")

In [ ]:
import numpy as np
def test_top():
    "lam1 and v1 are the top eigenvalue and unit eigenvector of W0, with v1 summing to a positive number"
    assert np.isclose(lam1, np.linalg.eigvalsh(W0).max())
    assert np.allclose(W0 @ v1, lam1 * v1) and v1.sum() > 0 and np.isclose(np.linalg.norm(v1), 1)

def test_tuned():
    "W_tuned has top eigenvalue exactly 1 and holds activity along v1"
    assert np.isclose(np.linalg.eigvalsh(W_tuned).max(), 1)
    assert np.allclose(held, held[0], rtol=1e-6)

def test_drift():
    "the measured time constant of the mistuned network matches the prediction within 5%"
    assert np.isclose(tau_pred, tau / (1 - 0.98))
    assert abs(tau_meas - tau_pred) / tau_pred < 0.05, f"measured {tau_meas:.2f} s"

test_top()
test_tuned()
test_drift()
print("All checks passed")

### Exercise: Part C: random networks and the circular law

`J` is a 300 by 300 matrix of independent random weights with variance 1/N. For each gain g in `gains`, compute the eigenvalues of `g * J`, store the largest absolute value in `radius[g]` and whether the rate network $\tau\dot{\mathbf{x}} = -\mathbf{x} + gJ\mathbf{x}$ is stable in `stable[g]`. Plot the eigenvalues in the complex plane.

In [ ]:
rng = np.random.default_rng(2006)
Nr = 300
J = rng.normal(size=(Nr, Nr)) / np.sqrt(Nr)
gains = [0.5, 0.9, 1.2]
radius, stable = {}, {}
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for g in gains:
    lam = ...
    radius[g] = ...
    stable[g] = ...
    ax.plot(lam.real, lam.imag, ".", ms=3, label=f"g = {g}")
ax.axvline(1, color="r", ls="--", lw=1); ax.set_aspect("equal"); ax.set_xlabel("Re λ"); ax.set_ylabel("Im λ"); ax.legend(frameon=False)
plt.show()
print({g: round(radius[g], 3) for g in gains}, stable)

In [ ]:
import numpy as np
def test_radius():
    "radius[g] is the largest |eigenvalue| of g J for each g, close to g itself"
    for g in gains:
        assert np.isclose(radius[g], np.abs(np.linalg.eigvals(g * J)).max())
        assert abs(radius[g] - g) / g < 0.1

def test_stable():
    "stable[g] says whether every eigenvalue of g J has real part below 1"
    assert stable[0.5] and stable[0.9] and not stable[1.2]

test_radius()
test_stable()
print("All checks passed")